# Etapa 3 — CTGAN sobre la capa emitida (dataset sintético v1)

Ejecuta el pre-registro del CTGAN y su adenda (`docs/reporte/adenda_preregistro_ctgan.md`, 17-nov-2026).
Corre de punta a punta con *Restart & Run All*; todas las cifras salen de `outputs/ctgan/`.

**Entorno.** `sdv`, `rdt` y `sdmetrics` exigen `pandas<3`; el repo fija `pandas==3.0.6`. Este notebook corre en un
entorno aparte (`requirements-ctgan.txt`) para no tocar el `.venv` de las Etapas 1–2.

**Desviaciones respecto al texto de la adenda, aceptadas por la autora el 17-nov-2026, antes de la corrida oficial:**

| # | Adenda | Lo que hace este notebook | Motivo |
|---|---|---|---|
| D1 | §4: rechazar «`canal_p2p` informado en filas cuyo `origen` no es P2P» | Rechaza `canal_p2p=True` cuando el `origen` no es compra / pago recurrente / ingreso no asalariado, el medio no es transferencia o billetera, o la categoría es `transferencias_p2p` | La regla R2 de la taxonomía define `canal_p2p=True` como pago a persona **con** propósito declarado; el P2P sin propósito lleva `False`. La regla literal rechaza el 7,06 % de la propia fuente |
| D2 | §3: «Referencia: elasticidad de reglas ≈ 1,008» | La referencia es la elasticidad **calculada aquí sobre la fuente** | 1,008 es del pipeline de Pancholi (`outputs/pancholi/hp1_elasticidad.csv`, conjunto A, etapa 2) y con otro estimador (monto medio por nivel de ingreso). No describe esta fuente |
| D3 | §1 «n = 122.211 completo» y §3 «contra un holdout de la fuente» | Holdout del 20 % de los **perfiles**; el CTGAN se entrena con el 80 % restante | Las dos frases no se cumplen a la vez con una sola corrida de reglas. Alternativa: entrenar con todo y generar el holdout con otra semilla de reglas |
| D4 | §3: métricas «relativas a piso y techo» | Relativo solo para Column Pair Trends; para Column Shapes y Overall, brecha absoluta al techo | El piso de marginales independientes conserva las marginales por construcción: en Column Shapes piso ≈ techo y el cociente se indetermina |

**Añadidos del 17-nov-2026 (antes de la corrida):** `signo` entra como columna discreta (es observable del clasificador y no
se deduce en `transferencias_p2p.indeterminada`); sexta regla dura: `signo` incompatible con el `origen`, salvo `transferencias_p2p.indeterminada`; criterio de conservación de la elasticidad por razón contra el train (§5); una tasa de rechazo
> 10 % es un resultado de fidelidad negativo que se reporta y se discute, y la corrida continúa (el v1 hace falta para el TSTR).

**Decisiones que la adenda no fija y aquí se toman (declaradas):** `dia_mes` y `mes` como categóricas; `categoria_n2`
vacía como nivel explícito; tamaño de la muestra = filas de la fuente; la marginal real de `categoria_n1` se impone por
cuotas sobre una muestra incondicional ya filtrada (es lo que hace SDV por dentro: `CTGANSynthesizer` no tiene muestreo
condicional y `sample_from_conditions` cae a muestreo por rechazo).

In [1]:
import hashlib, json, os, platform, random, subprocess, sys, time, gzip
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sdg").is_dir())
sys.path.insert(0, str(RAIZ / "src"))
from sdg import etapa3 as E

# ---- Parámetros pre-registrados (no cambiar tras ver resultados) ----
SEED = 42
EPOCAS = 300            # respaldo declarado: 150. Si se usa, queda en el manifest y se declara.
BATCH = 500
FRAC_HOLDOUT = 0.20     # D3: por perfil
TOPE_RECHAZO = 0.10
VERIFICAR_REPRO = True  # segunda corrida de entrenamiento para comparar SHA-256
MAX_LOTES = 20          # tope de lotes incondicionales para completar las cuotas de categoria_n1

# Modo de prueba (humo): ETAPA3_PRUEBA=1 → 2 épocas, salida en outputs/ctgan_prueba/. Nunca es la corrida oficial.
PRUEBA = os.environ.get("ETAPA3_PRUEBA", "0") == "1"
if PRUEBA:
    EPOCAS = int(os.environ.get("ETAPA3_PRUEBA_EPOCAS", "2"))
SALIDA = RAIZ / "outputs" / ("ctgan_prueba" if PRUEBA else "ctgan")
SALIDA.mkdir(parents=True, exist_ok=True)
FUENTE = RAIZ / "outputs" / "sintetico_reglas"
TAXONOMIA = RAIZ / "docs" / "taxonomia" / "taxonomia_gastos_colombia_v1.json"
print("raíz:", RAIZ.name, "| salida:", SALIDA.relative_to(RAIZ).as_posix(), "| épocas:", EPOCAS, "| PRUEBA:", PRUEBA)

raíz: tesis-sdg | salida: outputs/ctgan | épocas: 300 | PRUEBA: False


## 1. Datos de entrada (adenda §1)

In [2]:
man_fuente = json.loads((FUENTE / "manifest.json").read_text(encoding="utf-8"))
b_tx = gzip.open(FUENTE / "transacciones.csv.gz", "rb").read()
b_pf = (FUENTE / "perfiles_generados.csv").read_bytes()
sha_in = {"transacciones_csv": E.sha_bytes(b_tx), "perfiles_csv": E.sha_bytes(b_pf), "taxonomia_json": E.sha_bytes(TAXONOMIA.read_bytes())}
assert sha_in["transacciones_csv"] == man_fuente["sha256_transacciones_csv"], "transacciones.csv.gz no es la corrida del manifest de reglas"
assert sha_in["perfiles_csv"] == man_fuente["sha256_perfiles_csv"], "perfiles_generados.csv no es la corrida del manifest de reglas"

tx = pd.read_csv(FUENTE / "transacciones.csv.gz", low_memory=False)
perfiles = pd.read_csv(FUENTE / "perfiles_generados.csv")
tabla = E.construir_tabla(tx, perfiles)
assert len(tabla) == man_fuente["filas"]["emitida"] == 122_211
ANIO_MES = E.anio_por_mes(tabla)
PARES = E.pares_validos(TAXONOMIA, tabla)

train, holdout = E.partir_por_perfil(tabla, FRAC_HOLDOUT, SEED)
viol_fuente = E.violaciones(tabla, PARES, ANIO_MES)
assert not viol_fuente.any().any(), f"la fuente viola sus propias reglas duras: {viol_fuente.sum().to_dict()}"
print(f"fuente {len(tabla):,} filas / {tabla.perfil_id.nunique()} perfiles → train {len(train):,} ({train.perfil_id.nunique()} perfiles), "
      f"holdout {len(holdout):,} ({holdout.perfil_id.nunique()} perfiles); la fuente cumple las {len(E.REGLAS)} reglas duras (0 violaciones)")

fuente 122,211 filas / 1000 perfiles → train 96,386 (800 perfiles), holdout 25,825 (200 perfiles); la fuente cumple las 6 reglas duras (0 violaciones)


## 2. Entrenamiento (SDV `CTGANSynthesizer`, CPU) y verificación de reproducibilidad (adenda §5)

SDV no expone semilla para el ajuste: durante `fit` el modelo usa los generadores globales, así que se fijan
`random`, `numpy` y `torch` justo antes de cada entrenamiento. El muestreo posterior usa la semilla interna fija de SDV.

In [3]:
import inspect, warnings
import torch, sdv, ctgan, rdt, sdmetrics
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    from sdmetrics.reports.single_table import QualityReport

# El argumento para forzar CPU cambió de nombre entre versiones de SDV (cuda → enable_gpu)
ARG_CPU = {"enable_gpu": False} if "enable_gpu" in inspect.signature(CTGANSynthesizer.__init__).parameters else {"cuda": False}

SDTYPES = {**{c: "categorical" for c in E.DISCRETAS + E.DERIVADAS}, "canal_p2p": "boolean", "monto": "numerical", "ingreso_mensual": "numerical"}
META_TABLA = {"columns": {c: {"sdtype": SDTYPES[c]} for c in E.COLUMNAS_MODELO}}
metadata = Metadata.load_from_dict({"tables": {"emitida": META_TABLA}})
X_train = train[E.COLUMNAS_MODELO].copy()
N_SINT = len(tabla)


def fijar_semillas(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def entrenar(seed):
    fijar_semillas(seed)
    s = CTGANSynthesizer(metadata, epochs=EPOCAS, batch_size=BATCH, verbose=False, **ARG_CPU)
    t0 = time.time(); s.fit({"emitida": X_train})   # SDV 2.x: datos como {tabla: DataFrame}
    return s, time.time() - t0


def muestrear(s, n):
    m = s.sample(n)
    return (m["emitida"] if isinstance(m, dict) else m)[E.COLUMNAS_MODELO]


sint_A, seg_A = entrenar(SEED)
lote_A = muestrear(sint_A, N_SINT)
sha_A = E.sha_bytes(E.csv_bytes(lote_A))
print(f"corrida A: {seg_A/60:.1f} min, sha256 del primer lote {sha_A[:16]}…")

repro = {"verificada": VERIFICAR_REPRO, "sha256_lote_A": sha_A}
if VERIFICAR_REPRO:
    sint_B, seg_B = entrenar(SEED)
    lote_B = muestrear(sint_B, N_SINT)
    repro |= {"sha256_lote_B": E.sha_bytes(E.csv_bytes(lote_B)), "minutos_B": seg_B / 60}
    repro["bit_a_bit"] = repro["sha256_lote_B"] == sha_A
    if not repro["bit_a_bit"]:  # reproducibilidad estadística: diferencia entre corridas en las cantidades pre-registradas
        repro["estadistica"] = {
            "elasticidad_A": E.elasticidad(lote_A[lote_A.monto > 0]), "elasticidad_B": E.elasticidad(lote_B[lote_B.monto > 0]),
            "rechazo_A": float(E.violaciones(lote_A, PARES, ANIO_MES).any(axis=1).mean()),
            "rechazo_B": float(E.violaciones(lote_B, PARES, ANIO_MES).any(axis=1).mean())}
    del sint_B, lote_B
    print("reproducibilidad bit a bit:", repro["bit_a_bit"], "" if repro["bit_a_bit"] else repro["estadistica"])
sint_A.save(str(SALIDA / "ctgan_A.pkl"))

corrida A: 145.0 min, sha256 del primer lote 4371cac1b7e9870c…


reproducibilidad bit a bit: True 


## 3. Muestreo, reglas duras y tasa de rechazo (adenda §2 y §4)

La tasa de rechazo se mide sobre la muestra **incondicional** (lo que el modelo genera sin intervención). Luego se
toman, en orden de generación, las primeras filas válidas de cada `categoria_n1` hasta completar la cuota de la marginal
real del train.

In [4]:
objetivo = E.cuotas(train["categoria_n1"].value_counts(), N_SINT)
lotes, validas, n_gen, n_rech = [lote_A], [], 0, 0
por_regla = pd.Series(0, index=E.REGLAS)
cuenta = pd.Series(0, index=objetivo.index)
for i in range(MAX_LOTES):
    lote = lotes[0] if i == 0 else muestrear(sint_A, N_SINT)
    v = E.violaciones(lote, PARES, ANIO_MES)
    n_gen += len(lote); n_rech += int(v.any(axis=1).sum()); por_regla += v.sum()
    ok = lote.loc[~v.any(axis=1)]
    validas.append(ok)
    cuenta = cuenta.add(ok["categoria_n1"].value_counts(), fill_value=0).reindex(objetivo.index, fill_value=0)
    if (cuenta >= objetivo).all():
        break
pool = pd.concat(validas, ignore_index=True)
sint_v1 = (pool[pool.groupby("categoria_n1").cumcount() < pool["categoria_n1"].map(objetivo).fillna(0)]
           .sort_values(["mes", "dia_mes"], kind="mergesort").reset_index(drop=True))
deficit = (objetivo - sint_v1["categoria_n1"].value_counts().reindex(objetivo.index, fill_value=0)).clip(lower=0)

tasa = n_rech / n_gen
marg_modelo = lote_A["categoria_n1"].value_counts(normalize=True).reindex(objetivo.index, fill_value=0)
rechazo = {"filas_generadas": n_gen, "filas_rechazadas": n_rech, "tasa_rechazo": tasa, "tope_declarado": TOPE_RECHAZO,
           "cumple_tope": bool(tasa <= TOPE_RECHAZO), "resultado_fidelidad": "conforme" if tasa <= TOPE_RECHAZO else "NEGATIVO: supera el tope; se reporta y se discute", "por_regla": {k: int(x) for k, x in por_regla.items()},
           "tasa_por_regla": {k: float(x / n_gen) for k, x in por_regla.items()}, "lotes_usados": i + 1,
           "deficit_cuotas": {k: int(x) for k, x in deficit.items() if x > 0},
           "tv_marginal_n1_modelo_vs_train": float(0.5 * (marg_modelo - objetivo / objetivo.sum()).abs().sum())}
assert not E.violaciones(sint_v1, PARES, ANIO_MES).any().any()
print(f"tasa de rechazo {tasa:.2%} (tope {TOPE_RECHAZO:.0%}) → {'CUMPLE' if rechazo['cumple_tope'] else 'NO CUMPLE'}; dataset v1: {len(sint_v1):,} filas")
pd.Series(rechazo["tasa_por_regla"]).rename("tasa").to_frame()

tasa de rechazo 14.78% (tope 10%) → NO CUMPLE; dataset v1: 122,211 filas


,tasa
monto_no_positivo,0.000000
medio_imposible_para_cuenta,0.016733
par_n1_n2_inexistente,0.107822
canal_p2p_incoherente,0.002520
fecha_imposible,0.001109
signo_incompatible_con_origen,0.026610


## 4. Fidelidad: SDV `QualityReport` relativo a piso y techo (adenda §3)

Los tres conjuntos se comparan contra el **mismo** holdout. No hay umbral absoluto.

- **Column Pair Trends:** `relativo = (CTGAN − piso) / (techo − piso)`; 0 = no mejora a las marginales independientes,
  1 = tan cerca del holdout como el propio train.
- **Column Shapes y Overall (D4):** el piso reproduce las marginales del train por construcción, así que piso ≈ techo y el
  cociente no está definido. Se reporta la `brecha_al_techo = techo − CTGAN` (0 = marginales tan fieles como las del train).

In [5]:
piso = E.piso_marginales(train, len(sint_v1), SEED)
H = holdout[E.COLUMNAS_MODELO]


def calidad(sintetico):
    r = QualityReport(); r.generate(H, sintetico[E.COLUMNAS_MODELO], META_TABLA, verbose=False)
    p = r.get_properties().set_index("Property")["Score"]
    return {"overall": float(r.get_score()), "column_shapes": float(p["Column Shapes"]), "column_pair_trends": float(p["Column Pair Trends"])}, r


q, rep = {}, {}
for nombre, d in [("ctgan_v1", sint_v1), ("piso_marginales", piso), ("techo_train", train)]:
    q[nombre], rep[nombre] = calidad(d)
cal = pd.DataFrame(q).T
cal.loc["brecha_al_techo"] = cal.loc["techo_train"] - cal.loc["ctgan_v1"]
# D4: el relativo solo está definido donde piso y techo se separan (Column Pair Trends)
cal.loc["relativo"] = np.nan
cal.loc["relativo", "column_pair_trends"] = ((cal.loc["ctgan_v1", "column_pair_trends"] - cal.loc["piso_marginales", "column_pair_trends"])
                                             / (cal.loc["techo_train", "column_pair_trends"] - cal.loc["piso_marginales", "column_pair_trends"]))
rep["ctgan_v1"].get_details("Column Shapes").to_csv(SALIDA / "quality_column_shapes.csv", index=False, lineterminator="\n")
rep["ctgan_v1"].get_details("Column Pair Trends").to_csv(SALIDA / "quality_column_pair_trends.csv", index=False, lineterminator="\n")
cal.round(4)

,overall,column_shapes,column_pair_trends
ctgan_v1,0.9307,0.9346,0.9269
piso_marginales,0.8299,0.9799,0.6799
techo_train,0.9741,0.9803,0.9680
brecha_al_techo,0.0434,0.0457,0.0411
relativo,NaN,NaN,0.8575


## 5. Elasticidad monto–ingreso (adenda §3)

Pendiente OLS de `log(monto)` sobre `log(ingreso_mensual)` a nivel transacción, igual en todos los conjuntos. IC 95 %
por bootstrap: de **perfiles** en la fuente (1.000 ingresos distintos, no 122.211) y de filas en los conjuntos sin perfil.

**Criterio (fijado el 17-nov-2026, antes de la corrida).** Con `r = ê_ctgan / ê_train`:
- **CONSERVA:** `ê_ctgan` positiva, IC excluye 0 y r ≥ 0,70 (incluye r > 1).
- **ATENÚA:** positiva, IC excluye 0 y 0,30 ≤ r < 0,70.
- **NO CONSERVA:** r < 0,30, o `ê_ctgan` ≤ 0, o el IC incluye 0.

La comparación con Pancholi (−0,003) es cualitativa: su IC era entre 10 semillas, otra fuente de varianza.

In [6]:
conj = [("fuente_completa", tabla, "perfil_id"), ("train", train, "perfil_id"), ("holdout", holdout, "perfil_id"),
        ("ctgan_v1", sint_v1, None), ("piso_marginales", piso, None)]
ela = pd.DataFrame([{"conjunto": n, "n": len(d), "elasticidad": E.elasticidad(d),
                     **dict(zip(["ic95_inf", "ic95_sup"], E.ic_elasticidad(d, SEED, 1000, c))),
                     "ic": "bootstrap de perfiles" if c else "bootstrap de filas"} for n, d, c in conj]).set_index("conjunto")
ela["razon_vs_train"] = ela["elasticidad"] / ela.loc["train", "elasticidad"]
veredicto, r = E.veredicto_elasticidad(ela.loc["ctgan_v1", "elasticidad"], (ela.loc["ctgan_v1", "ic95_inf"], ela.loc["ctgan_v1", "ic95_sup"]),
                                      ela.loc["train", "elasticidad"])
print(f"ê_train = {ela.loc['train', 'elasticidad']:.3f}; ê_ctgan = {ela.loc['ctgan_v1', 'elasticidad']:.3f} "
      f"[{ela.loc['ctgan_v1', 'ic95_inf']:.3f}; {ela.loc['ctgan_v1', 'ic95_sup']:.3f}]; r = {r:.2f} → veredicto: {veredicto}")
ela.round(4)

ê_train = 0.331; ê_ctgan = 0.452 [0.441; 0.463]; r = 1.37 → veredicto: CONSERVA


,n,elasticidad,ic95_inf,ic95_sup,ic,razon_vs_train
conjunto,,,,,,
fuente_completa,122211,0.3474,0.2918,0.4045,bootstrap de perfiles,1.0506
train,96386,0.3307,0.2674,0.3885,bootstrap de perfiles,1.0000
holdout,25825,0.4072,0.2628,0.5519,bootstrap de perfiles,1.2314
ctgan_v1,122211,0.4515,0.4408,0.4627,bootstrap de filas,1.3654
piso_marginales,122211,-0.0047,-0.0152,0.0063,bootstrap de filas,-0.0142


### Exploratorio (no pre-registrado; no entra al veredicto)

La pendiente global mezcla filas de ingreso (monto ≈ ingreso, pendiente ≈ 1) con compras. Un CTGAN puede acertar la
global sin conservar la relación dentro de cada tipo de movimiento, así que se muestra por `origen`. También se cuenta
cuántas filas del v1 traen un par (`origen`, `categoria_n1`) que la fuente nunca emite: incoherencia estructural que
las reglas duras pre-registradas no cubren.

In [7]:
por_origen = pd.DataFrame({n: d.groupby("origen").apply(E.elasticidad, include_groups=False) for n, d in [("train", train), ("ctgan_v1", sint_v1)]})
por_origen["n_train"] = train["origen"].value_counts(); por_origen["n_ctgan"] = sint_v1["origen"].value_counts()
par_fuente = set(map(tuple, tabla[["origen", "categoria_n1"]].drop_duplicates().to_numpy()))
frac_par_nuevo = float((~pd.MultiIndex.from_frame(sint_v1[["origen", "categoria_n1"]]).isin(list(par_fuente))).mean())
print(f"filas del v1 con par (origen, categoria_n1) inexistente en la fuente: {frac_par_nuevo:.2%}")
por_origen.round(3)

filas del v1 con par (origen, categoria_n1) inexistente en la fuente: 2.14%


,train,ctgan_v1,n_train,n_ctgan
origen,,,,
compra,0.321,0.428,54601,69185
ingreso_no_asalariado,0.979,0.492,2621,3558
nomina,0.997,0.438,1581,1900
p2p_sin_proposito,0.882,0.266,6099,7745
pago_recurrente,0.217,0.210,6496,8170
pago_tarjeta,0.448,0.421,472,941
recarga_billetera,0.427,0.370,9484,11708
retiro,0.341,0.367,15032,19004


## 6. Salidas y manifest (adenda §5 y «Salidas»)

In [8]:
salida = sint_v1.copy()
salida["anio"] = salida["mes"].map(ANIO_MES)
salida["fecha"] = pd.to_datetime(dict(year=salida.anio, month=salida.mes, day=salida.dia_mes)).dt.strftime("%Y-%m-%d")
salida["categoria_n2"] = salida["categoria_n2"].replace(E.SIN_N2, "")
salida = salida[["fecha"] + E.COLUMNAS_MODELO]
b_v1 = E.csv_bytes(salida)
with open(SALIDA / "sintetico_v1.csv.gz", "wb") as fh, gzip.GzipFile(fileobj=fh, mode="wb", mtime=0, filename="") as gz:
    gz.write(b_v1)
cal.to_csv(SALIDA / "quality_report.csv", lineterminator="\n")
ela.to_csv(SALIDA / "elasticidad.csv", lineterminator="\n")
por_origen.to_csv(SALIDA / "exploratorio_elasticidad_por_origen.csv", lineterminator="\n")
(SALIDA / "rechazo.json").write_text(json.dumps(rechazo, indent=1, ensure_ascii=False), encoding="utf-8")


def commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=RAIZ, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None


manifest = {
    "notebook": "notebooks/03_ctgan_etapa3.ipynb", "adenda": "docs/reporte/adenda_preregistro_ctgan.md", "modo_prueba": PRUEBA,
    "commit_base": commit(), "semilla": SEED, "semilla_muestreo_sdv": "FIXED_RNG_SEED interna de SDV",
    "parametros": {"epocas": EPOCAS, "batch_size": BATCH, "cuda": False, "frac_holdout_perfiles": FRAC_HOLDOUT, "n_sintetico": N_SINT,
                   "columnas": E.COLUMNAS_MODELO, "sdtypes": SDTYPES, "resto_ctgan": "valores por defecto de SDV"},
    "desviaciones_adenda": ["D1 regla canal_p2p según R2", "D2 referencia de elasticidad = fuente (no 1,008)", "D3 holdout 20 % de perfiles; train = 80 %",
                            "D4 relativo solo en Column Pair Trends; brecha al techo en Column Shapes y Overall"],
    "versiones": {"python": platform.python_version(), "torch": torch.__version__, "sdv": sdv.__version__, "ctgan": ctgan.__version__,
                  "rdt": rdt.__version__, "sdmetrics": sdmetrics.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "entorno": {"plataforma": platform.platform(), "dispositivo": "cpu", "gpu_disponible": bool(torch.cuda.is_available()),
                "hilos_torch": torch.get_num_threads(), "minutos_entrenamiento_A": seg_A / 60},
    "filas": {"fuente": len(tabla), "train": len(train), "holdout": len(holdout), "sintetico_v1": len(salida)},
    "sha256_entradas": sha_in, "reproducibilidad": repro, "rechazo": rechazo,
    "calidad": cal.to_dict(orient="index"), "elasticidad": ela.drop(columns="ic").to_dict(orient="index"), "veredicto_elasticidad": {"r": r, "umbral_conserva": E.UMBRAL_CONSERVA, "umbral_atenua": E.UMBRAL_ATENUA, "veredicto": veredicto},
    "exploratorio": {"frac_par_origen_n1_inexistente": frac_par_nuevo},
    "sha256_salidas": {"sintetico_v1_csv": E.sha_bytes(b_v1), **{f.name: E.sha_bytes(f.read_bytes()) for f in sorted(SALIDA.glob("*.csv")) + [SALIDA / "rechazo.json"]}},
}
(SALIDA / "manifest.json").write_text(json.dumps(manifest, indent=1, ensure_ascii=False, default=float), encoding="utf-8")
print("escrito en", SALIDA.relative_to(RAIZ).as_posix(), "→", sorted(f.name for f in SALIDA.iterdir()))

escrito en outputs/ctgan → ['ctgan_A.pkl', 'elasticidad.csv', 'exploratorio_elasticidad_por_origen.csv', 'manifest.json', 'quality_column_pair_trends.csv', 'quality_column_shapes.csv', 'quality_report.csv', 'rechazo.json', 'sintetico_v1.csv.gz']
